# 2 · Building and running a graph

A graph is its nodes. Each placed node says, input by input, where its value comes from: an **edge** from another node's output (`From`), a value the author **typed in** (`Static`), or nothing, and then the input's default applies. There is no separate list of edges.

- `Graph(nodes=[GraphNode(id=, type=, version=, bindings=)])` — the record a host saves.
- `CompiledGraph.from_graph(graph, registry)` — checks the graph and answers questions about it. It never raises for a fault in the graph: what is wrong is a `Problem` on the result.
- `execute(compiled)` — runs it, as a stream of events. In a notebook the kernel already runs an event loop, so iterate the stream or `await run(compiled)`, which returns the event the leg ended on; from a script, `run_sync(compiled)` is the same call.

In [ ]:
from typing import Annotated

from conductor import CompiledGraph, From, Graph, GraphNode, NodeDefinition, NodeRegistry, Param, Ref, Result, Series, Static, execute, run
from conductor.widgets import TextWidget
from conductor_nodes.types import Text


class Echo(NodeDefinition):
    id = "echo"
    title = "Echo"
    description = "Returns its input"
    category = "text"

    def run(self, text: Annotated[Text, Param(title="Input", widget=TextWidget())]) -> Annotated[Text, Result(title="Output")]:
        return text


class Upper(NodeDefinition):
    id = "upper"
    title = "Uppercase"
    description = "Uppercases a text"
    category = "text"

    def run(self, text: Annotated[Text, Param(title="Input", widget=TextWidget())]) -> Annotated[Text, Result(title="Output")]:
        return Text(text.upper())


class Pair(NodeDefinition):
    id = "pair"
    title = "Pair"
    description = "Joins two texts"
    category = "text"

    def run(
        self,
        a: Annotated[Text, Param(title="A", widget=TextWidget())],
        b: Annotated[Text, Param(title="B", widget=TextWidget())],
        separator: Annotated[Text, Param(title="Separator", widget=TextWidget())] = Text(" + "),
    ) -> Annotated[Text, Result(title="Result")]:
        return Text(f"{a}{separator}{b}")


registry = NodeRegistry()
for cls in (Echo, Upper, Pair):
    registry.register(cls)

## Build the graph

```text
  hello ──> shout ──┐
                    ├──> pair
  world ────────────┘
```

`hello` and `world` read nothing, so they start together. `shout` waits for `hello`, and `pair` for both. A node with one output names it `result`, so an edge to it is `Ref("hello", "result")`.

In [ ]:
graph = Graph(nodes=[
    GraphNode(id="hello", type="echo", version=1, bindings={"text": Static("hello")}),
    GraphNode(id="world", type="echo", version=1, bindings={"text": Static("world")}),
    GraphNode(id="shout", type="upper", version=1, bindings={"text": From("hello.result")}),
    GraphNode(id="pair", type="pair", version=1, bindings={
        "a": From("shout.result"),
        "b": From("world.result"),
        "separator": Static(" & "),
    }),
])

compiled = CompiledGraph.from_graph(graph, registry)
print("runnable:", compiled.is_runnable)
print("order:   ", compiled.execution_order)

## Ask the compiled graph

A compiled graph is asked, not traversed. At the scale of one node, `compiled.node(node_id)`; at the scale of one field, `compiled.field(ref)`.

In [ ]:
print("pair.a comes from:", compiled.field(Ref("pair", "a")).binding)
print("pair.a receives:  ", compiled.field(Ref("pair", "a")).receives)
print("pair.a is:        ", compiled.field(Ref("pair", "a")).type.describe())

## Draw it

`compiled.render()` is the graph as a Mermaid flowchart: a box per node, and an arrow per edge labelled with how its input receives it — nothing for a value that reaches every row, `per row of <index>` where a node runs once per row, `whole` where it takes the series entire. Paste it into anything that renders Mermaid.

In [ ]:
print(compiled.render())

## What is wrong is a problem, not an exception

An edge to a node that is not there, or a value of the wrong type, still compiles. The result says what is wrong, on which node, with a stable `code` a host can key on. `execute` refuses a graph with a fatal problem.

In [ ]:
broken = CompiledGraph.from_graph(
    Graph(nodes=[GraphNode(id="shout", type="upper", version=1, bindings={"text": From("ghost.result")})]),
    registry,
)
print("runnable:", broken.is_runnable)
for problem in broken.problems:
    print(f"  {problem.code} on {problem.node_id}: {problem.message}")

## Run it and read the ending

`await run(compiled)` drains the event stream and returns the event the leg ended on; every ending carries the run's `state`, and `state.results(compiled)` reads every node's values out of it, `{node_id: {output_name: value}}`. From a script, `run_sync(compiled)` is the same call.

In [ ]:
ending = await run(compiled)
results = ending.state.results(compiled)
for node_id, outputs in results.items():
    print(f"  {node_id}: {outputs['result']!r}")

## Fill its inputs, read its outputs

A graph's interface is what it takes and returns: here `hello.text` and `world.text`, the inputs of the nodes nothing feeds, and `pair.result`, the output nothing reads. `compiled.with_inputs(...)` is a copy with some inputs filled — by bare name when only one input has it, else by address, as here where two inputs are called `text` — and `run` takes the copy like any other. `state.outputs(compiled)` reads what the graph returns, keyed by address.

In [ ]:
print("takes:  ", [str(i.name) for i in compiled.interface.inputs])
print("returns:", [str(o.name) for o in compiled.interface.outputs])

ready = compiled.with_inputs(**{"hello.text": "goodbye"})
ending = await run(ready)
ending.state.outputs(ready)

## Watch the events

Iterate the stream to see what the engine does as it happens. `hello` and `world` both start before either completes. The last event is the ending, and it carries the run's `state`, which a host keeps to start a later leg from.

In [ ]:
async for event in execute(compiled):
    match event.type:
        case "node_start":
            print(f"  start     {event.node_id}")
        case "node_complete":
            print(f"  complete  {event.node_id}: {event.result['result']!r}")
        case "graph_complete":
            print(f"  done      {sorted(event.state.results(compiled))}")
        case other:
            print(f"  {other}")

## Once per row

Type a list into an input declared for one value and the node runs once per value; so does everything downstream of it, row by row. An input declared `Series[X]` receives the whole series in one call instead. That difference is all there is to iteration.

In [ ]:
class JoinAll(NodeDefinition):
    id = "join-all"
    title = "Join all"
    description = "Joins every text of a series"
    category = "text"

    def run(self, texts: Annotated[Series[Text], Param(title="Texts")]) -> Annotated[Text, Result(title="Result")]:
        return Text(" ".join(texts))


registry.register(JoinAll)

rows = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="shout", type="upper", version=1, bindings={"text": Static(["one", "two", "three"])}),
        GraphNode(id="all", type="join-all", version=1, bindings={"texts": From("shout.result")}),
    ]),
    registry,
)
print("shout runs per row of:", rows.node("shout").iterates_on)

async for event in execute(rows):
    if event.type == "node_progress":
        print(f"  {event.node_id}: {event.done} of {event.total}")
    elif event.type == "graph_complete":
        results = event.state.results(rows)
        print("  shout:", list(results["shout"]["result"]))
        print("  all:  ", results["all"]["result"])

## Save it

A `Graph` saves itself as YAML or JSON and reads back what it wrote. An edge is stored as its address, `"node.field"`.

In [ ]:
text = graph.to_yaml()
print(text)
assert Graph.from_yaml(text) == graph